In [ ]:
# PANDAS IS FOR DATA WRANGLING
import pandas as pd

# PLOTLY MAKES INTERACTIVE CHARTS
import plotly.express as px

# METRICS HELP US SCORE OUR MODEL
from sklearn import metrics

# HELP US SPLIT OUR DATA INTO TESTING AND TRAINING
from sklearn.model_selection import train_test_split

# Good ol statsmodels
import statsmodels.api as sm

# Specific root mean squared error for stats models
from statsmodels.tools.eval_measures import rmse

####################################################################################
## LOADING OUR DATA
####################################################################################
df = pd.read_csv('data/baseball.csv')

# Make data readable 
# Spell out the abbreviated column names so the data is easier to read
column_names = {
    'RS': 'runs_scored',
    'RA': 'runs_allowed',
    'W': 'wins',
    'OBP': 'on_base_percentage',
    'SLG': 'slugging_percentage',
    'BA': 'batting_average',
    'G': 'games',
    'OOBP': 'opponent_obp',
    'OSLG': 'opponent_slg',
    'RankSeason': 'rank_season',
    'RankPlayoffs': 'rank_playoffs',
}
df = df.rename(columns=column_names)

# Make every column label lower case (e.g. Team -> team, Year -> year)
df.columns = df.columns.str.lower()


####################################################################################
## ENGINEERING OUR DATA
####################################################################################
df['runs_diff'] = df['runs_scored'] - df['runs_allowed']

# what_we_are_trying_to_predict
dependent_variable = 'wins'

# what we are using to predict wins aka data_we_have_related_to_games
independent_variables = ['runs_diff']

# Define the independent and dependent variables, plus a constant for the intercept
X = sm.add_constant(df[independent_variables])
y = df[dependent_variable]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


print('Predicting:', dependent_variable)
print('Using:', independent_variables)

# Build the model on the training data
model = sm.OLS(y_train, X_train).fit()

# Score a model built on held-out data
print('\nEVAL ON TESTING DATA')
y_pred = model.predict(X_test)

# model.summary2()

print('Adjusted R-squared:', model.rsquared_adj.round(4))
print('Root Mean Squared Error (RMSE) on Test Data:', rmse(y_test, y_pred).round(4))
print('Model Parameters:\n', model.params.round(4))
print('P-values:\n', model.pvalues.round(4))


Predicting: wins
Using: ['runs_diff']

EVAL ON TESTING DATA
Adjusted R-squared: 0.8822
Root Mean Squared Error (RMSE) on Test Data: 4.0586
Model Parameters:
 const        80.7919
runs_diff     0.1057
dtype: float64
P-values:
 const        0.0
runs_diff    0.0
dtype: float64


# We know we need 95 wins... 

## How dow we figure out how many runs_diff will get us that...?

### Set wins = 95 and solve the line for run differential:

$$95 = 80.88 + 0.1058 \times \text{runs\_diff} \quad\Rightarrow\quad \text{runs\_diff} = \frac{95 - 80.88}{0.1058} \approx 133$$

**The A's need to outscore their opponents by roughly 133–135 runs over the season.** DePodesta reached essentially the same number. Now the problem is concrete: build a roster that scores about 135 more runs than it allows.